# Final Project Part 2 — Data Viz for Experts
## Chicago Food Inspections Dataset
**Student Names :** Gayatri Mahindrakar, Harisankar Kartha (Group 5)
---

## Dataset Information
- **Name:** Chicago Food Inspections Dataset
- **Source:** City of Chicago Data Portal, maintained by the Chicago Department of Public Health (CDPH)
- **Direct CSV URL:** https://data.cityofchicago.org/api/views/4ijn-s7e5/rows.csv?accessType=DOWNLOAD
- **Large Data Plan:** This dataset is ~323 MB on disk and exceeds GitHub's 100 MB file 
  upload limit. Our plan varies by part:
  - **Parts 1 & 2:** Data is loaded dynamically at runtime directly from the City of 
    Chicago Open Data API URL above. No local file upload is needed and any grader can 
    run the notebook without a local copy of the data.
  - **Part 3:** Since Part 3 requires a public-facing web deployment, loading 323 MB 
    on every page visit would be impractical. Instead, we will pre-compute the two 
    aggregated dataframes used in our visualizations — `driver_data` (24 rows) and 
    `yearly_pass` (128 rows) — and host them as lightweight CSVs on GitHub. These 
    derived datasets are well under 1 KB combined, fall comfortably within GitHub's 
    file size limits, and faithfully represent all analysis performed on the full dataset.

In [1]:
import pandas as pd
import numpy as np
import altair as alt
import warnings
warnings.filterwarnings('ignore')

# Enable Altair renderer for JupyterLab
alt.renderers.enable('default')

# Allow Altair to handle large datasets (disable 5000 row limit)
alt.data_transformers.disable_max_rows()

# Load full dataset directly from City of Chicago open data portal
# Dataset is ~323 MB on disk — loaded via URL, not uploaded locally
URL = "https://data.cityofchicago.org/api/views/4ijn-s7e5/rows.csv?accessType=DOWNLOAD"
df = pd.read_csv(URL)

print("Data loaded successfully!")
print(f"Shape: {df.shape}")

Data loaded successfully!
Shape: (309328, 17)


In [2]:
# Parse inspection date column and extract Year
df['Inspection Date'] = pd.to_datetime(df['Inspection Date'])
df['Year'] = df['Inspection Date'].dt.year

# Keep only the three meaningful inspection results
# Filter out junk Risk values ('All', NaN)
# Remove 2026 since year is still in progress (partial data)
df_filtered = df[
    df['Results'].isin(['Pass', 'Fail', 'Pass w/ Conditions']) &
    df['Risk'].isin(['Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)']) &
    df['Facility Type'].notna() &
    (df['Year'] >= 2010) &
    (df['Year'] <= 2025)
].copy()

# Keep only the top 8 facility types for readability in the driver plot
top_facilities = df_filtered['Facility Type'].value_counts().head(8).index.tolist()
df_filtered = df_filtered[df_filtered['Facility Type'].isin(top_facilities)]

print(f"Filtered dataset shape: {df_filtered.shape}")
print(f"\nTop 8 facility types included:")
for f in top_facilities:
    print(f"  - {f}")

Filtered dataset shape: (246990, 18)

Top 8 facility types included:
  - Restaurant
  - Grocery Store
  - School
  - Children's Services Facility
  - Bakery
  - Daycare Above and Under 2 Years
  - Daycare (2 - 6 Years)
  - Long Term Care


In [3]:
# -------------------------------------------------------
# DRIVER PLOT DATA
# Count of inspections by Facility Type and Result
# -------------------------------------------------------
driver_data = (
    df_filtered
    .groupby(['Facility Type', 'Results'])
    .size()
    .reset_index(name='Count')
)

driver_data['Total per Facility'] = driver_data.groupby(
    'Facility Type')['Count'].transform('sum')
driver_data['Percentage'] = (
    driver_data['Count'] / driver_data['Total per Facility'] * 100
).round(1)

# -------------------------------------------------------
# DRIVEN PLOT DATA
# Pass rate by Year and Facility Type
# -------------------------------------------------------

# Total inspections per facility type per year
yearly_total = (
    df_filtered
    .groupby(['Facility Type', 'Year'])
    .size()
    .reset_index(name='Total')
)

# Pass-only counts per facility type per year
yearly_facility = (
    df_filtered
    .groupby(['Facility Type', 'Year', 'Results'])
    .size()
    .reset_index(name='Count')
)

yearly_pass = yearly_facility[yearly_facility['Results'] == 'Pass'].copy()

# Merge to compute pass rate
yearly_pass = yearly_pass.merge(yearly_total, on=['Facility Type', 'Year'])
yearly_pass['Pass Rate'] = (yearly_pass['Count'] / yearly_pass['Total'] * 100).round(2)

# Keep only the columns needed for the driven plot
yearly_pass = yearly_pass[['Facility Type', 'Year', 'Pass Rate']]

print("Driver data sample:")
print(driver_data.head(6))
print(f"\nDriver data shape: {driver_data.shape}")
print("\nDriven data sample:")
print(yearly_pass.head(6))
print(f"\nDriven data shape: {yearly_pass.shape}")

Driver data sample:
                  Facility Type             Results  Count  \
0                        Bakery                Fail    934   
1                        Bakery                Pass   2241   
2                        Bakery  Pass w/ Conditions    611   
3  Children's Services Facility                Fail   1222   
4  Children's Services Facility                Pass   4683   
5  Children's Services Facility  Pass w/ Conditions    886   

   Total per Facility  Percentage  
0                3786        24.7  
1                3786        59.2  
2                3786        16.1  
3                6791        18.0  
4                6791        69.0  
5                6791        13.0  

Driver data shape: (24, 5)

Driven data sample:
  Facility Type  Year  Pass Rate
0        Bakery  2010      69.06
1        Bakery  2011      69.93
2        Bakery  2012      67.84
3        Bakery  2013      73.86
4        Bakery  2014      67.67
5        Bakery  2015      70.25

Driven data 

In [4]:
# # NOTE: This cell was an attempt to locate and fix an error with the visualization but turned out to be unnecessary.
# # The actual fix was .resolve_scale(color='independent') in the dashboard cell.
# # Keeping here as per rubric instructions (do not delete, comment out).

# # Total inspections per Facility Type per Year
# yearly_total = (
#     df_filtered
#     .groupby(['Facility Type', 'Year'])
#     .size()
#     .reset_index(name='Total')
# )

# # Pass count per Facility Type per Year
# yearly_pass_count = (
#     df_filtered[df_filtered['Results'] == 'Pass']
#     .groupby(['Facility Type', 'Year'])
#     .size()
#     .reset_index(name='PassCount')
# )

# # Fail count per Facility Type per Year
# yearly_fail_count = (
#     df_filtered[df_filtered['Results'] == 'Fail']
#     .groupby(['Facility Type', 'Year'])
#     .size()
#     .reset_index(name='FailCount')
# )

# # Pass w/ Conditions count per Facility Type per Year
# yearly_pwc_count = (
#     df_filtered[df_filtered['Results'] == 'Pass w/ Conditions']
#     .groupby(['Facility Type', 'Year'])
#     .size()
#     .reset_index(name='PwCCount')
# )

# # Merge all together
# yearly_combined = yearly_total.copy()
# yearly_combined = yearly_combined.merge(yearly_pass_count, on=['Facility Type', 'Year'], how='left')
# yearly_combined = yearly_combined.merge(yearly_fail_count, on=['Facility Type', 'Year'], how='left')
# yearly_combined = yearly_combined.merge(yearly_pwc_count, on=['Facility Type', 'Year'], how='left')
# yearly_combined = yearly_combined.fillna(0)
# yearly_combined['Pass Rate'] = (yearly_combined['PassCount'] / yearly_combined['Total'] * 100).round(2)

# # Driver data — total counts by Facility Type and Result (for stacked bar)
# driver_data = (
#     df_filtered
#     .groupby(['Facility Type', 'Results'])
#     .size()
#     .reset_index(name='Count')
# )

# # Driven data — pass rate by Facility Type and Year
# yearly_pass = yearly_combined[['Facility Type', 'Year', 'Pass Rate']].copy()

# print("Driver data shape:", driver_data.shape)
# print("Driven data shape:", yearly_pass.shape)
# print("\nSample driven data:")
# print(yearly_pass.head(6))

In [18]:
driver_records = driver_data.copy()
driven_records = yearly_pass.copy()

# -------------------------------------------------------
# COLOR SCALES 
# -------------------------------------------------------
result_colors = alt.Scale(
    domain=['Pass', 'Fail', 'Pass w/ Conditions'],
    range=['#59A14F', '#C0392B', '#F4D03F']
)

facility_colors = alt.Scale(
    domain=[
        'Restaurant',
        'Grocery Store', 
        'School',
        "Children's Services Facility",
        'Bakery',
        'Daycare Above and Under 2 Years',
        'Daycare (2 - 6 Years)',
        'Long Term Care'
    ],
    range=[
        '#4E79A7',
        '#F28E2B',
        '#59A14F',
        '#E15759',
        '#76B7B2',
        '#EDC948',
        '#B07AA1',
        '#FF9DA7'
    ]
)

selection = alt.selection_point(
    fields=['Facility Type'],
    empty='all'
)

# -------------------------------------------------------
# 1. DRIVER PLOT
# -------------------------------------------------------
driver = (
    alt.Chart(driver_records)
    .mark_bar()
    .encode(
        x=alt.X(
            'Facility Type:N',
            sort='-y',
            title='Facility Type',
            axis=alt.Axis(labelAngle=-35, labelLimit=200, titlePadding=25)
        ),
        y=alt.Y(
            'sum(Count):Q',
            title='Total Number of Inspections',
            axis=alt.Axis(titlePadding=20)
        ),
        color=alt.Color(
            'Facility Type:N',
            scale=facility_colors,
            legend=None
        ),
        opacity=alt.condition(selection, alt.value(1.0), alt.value(0.2)),
        tooltip=[
            alt.Tooltip('Facility Type:N', title='Facility Type'),
            alt.Tooltip('sum(Count):Q', title='Total Inspections', format=',')
        ]
    )
    .add_params(selection)
    .properties(
        width=550,
        height=300,
        title=alt.TitleParams(
            text='Total Inspections by Facility Type (Driver Plot)',
            subtitle='Click a bar to filter the charts below',
            offset=25
        )
    )
)

# -------------------------------------------------------
# 2. DONUT CHART
# -------------------------------------------------------
donut = (
    alt.Chart(driver_records)
    .mark_arc(innerRadius=60)
    .encode(
        theta=alt.Theta('Count:Q'),
        color=alt.Color('Results:N', scale=result_colors, title='Result'),
        tooltip=[
            alt.Tooltip('Results:N', title='Result'),
            alt.Tooltip('Count:Q', title='Count', format=','),
            alt.Tooltip('Percentage:Q', title='% of Total', format='.1f')
        ]
    )
    .transform_filter(selection)
    .properties(
        width=250,
        height=250,
        title=alt.TitleParams(
            text='Result Proportions (Driven Plot)',
            subtitle=' ',
            anchor='middle',
            offset=20
        )
    )
)

# -------------------------------------------------------
# 3. LINE CHART
# -------------------------------------------------------
driven_line = (
    alt.Chart(driven_records)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X(
            'Year:O',
            title='Year',
            axis=alt.Axis(labelAngle=-45, titlePadding=15)
        ),
        y=alt.Y(
            'Pass Rate:Q',
            title='Pass Rate (%)',
            scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(tickCount=10)
        ),
        color=alt.Color(
            'Facility Type:N',
            scale=facility_colors,
            title='Facility Type',
            legend=alt.Legend(offset=30)
        ),
        tooltip=[
            alt.Tooltip('Facility Type:N', title='Facility Type'),
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Pass Rate:Q', title='Pass Rate (%)', format='.1f')
        ]
    )
    .transform_filter(selection)
    .properties(
        width=800,
        height=300,
        title=alt.TitleParams(
            text='Pass Rate Trend Over Time (Driven Plot)',
            subtitle='Historical pass rates for selection',
            offset=20
        )
    )
)

# -------------------------------------------------------
# COMBINE DASHBOARD
# -------------------------------------------------------
top_row = alt.hconcat(driver, donut, spacing=60).resolve_scale(color='independent')

dashboard = (
    alt.vconcat(top_row, driven_line, spacing=80)
    .resolve_scale(color='independent')
    .configure_title(
        fontSize=18,
        subtitleFontSize=13,
        subtitleColor='gray',
        font='Arial',
        anchor='start',
        color='#2c3e50'
    )
    .configure_axis(
        labelFontSize=12,
        titleFontSize=14,
        titleFontWeight='bold',
        grid=True,
        gridColor='#E6E6E6'
    )
    .configure_legend(
        labelFontSize=12,
        titleFontSize=14
    )
    .configure_view(
    fill='#F7F7F7',
    stroke=None
    )
)

dashboard

alt.VConcatChart(...)

In [6]:
# DEBUG CELL — verifying results, kept for transparency (commented out as per rubric)
# print("yearly_pass shape:", yearly_pass.shape)
# print("\nSample rows:")
# print(yearly_pass.head(10))
# print("\nFacility types in yearly_pass:")
# print(yearly_pass['Facility Type'].unique())
# print("\nYear range:")
# print(yearly_pass['Year'].min(), "to", yearly_pass['Year'].max())
# print("\nPass Rate range:")
# print(yearly_pass['Pass Rate'].min(), "to", yearly_pass['Pass Rate'].max())

## How to Use This Dashboard (Non-Expert Guide)

This interactive dashboard explores over **246,000 Chicago food establishment 
inspections** conducted by the Chicago Department of Public Health (CDPH) between 
**2010 and 2025**. It focuses on the eight most commonly inspected facility types — 
including restaurants, grocery stores, schools, bakeries, and daycare centers — to 
help users understand inspection outcomes and how they change over time. The dashboard 
is designed to be intuitive and interactive, allowing public health analysts, 
policymakers, and general citizens interested in food safety trends to explore the data 
at different levels of detail.

---

## Dashboard Components

The dashboard consists of **three linked visual components**:

- **Top-Left Bar Chart (Driver Plot):**  
  Shows the total number of inspections for each of the eight most frequently 
  inspected facility types. Click on any bar (e.g., *Restaurant* or *Grocery Store*) 
  to filter both charts to the right and below. Clicking the same bar again, or 
  clicking on empty space, resets the dashboard back to showing all facility types.

- **Top-Right Donut Chart (Driven Plot):**  
  Displays the proportional breakdown of inspection outcomes — *Pass*, *Fail*, and 
  *Pass with Conditions* — for whichever facility type is currently selected in the 
  bar chart. In the default (unselected) state, it shows the aggregate breakdown 
  across all eight facility types combined. Hover over each segment to see the exact 
  count and percentage.

- **Bottom Line Chart (Driven Plot):**  
  Tracks how the pass rate — the percentage of inspections resulting in a clean 
  *Pass* — has changed year by year from **2010 to 2025** for the selected facility 
  type. When no selection is active, all eight facility types are shown simultaneously, 
  allowing direct comparison of trends. The dramatic synchronized dip visible around 
  **2018–2019** reflects the CDPH's redefinition of violation categories, while the 
  smaller dip in **2020–2021** corresponds to the COVID-19 pandemic.

---

## How to Interact

- **Click** any bar in the top-left chart to filter the donut and line chart  
- **Hover** over any chart element to see detailed values and percentages  
- **Click again** or click outside any bar to reset all charts to the full view  

---

## Key Insights from the Dashboard

- **Restaurants dominate inspections**, accounting for ~175,000 out of 246,990 
  filtered records — nearly 6× more than the next largest group (Grocery Stores 
  at ~31,000)
- Across all facility types in aggregate, **Pass results make up roughly half** of 
  all outcomes, with *Fail* and *Pass with Conditions* making up the remaining share, 
  as visible in the default state of the donut chart
- A **sharp, universal drop in pass rates occurred in 2018–2019** across all facility 
  types, directly caused by the CDPH redefining its violation categories — the most 
  dramatic structural shift visible in the line chart
- The **2018 crash hit Restaurants and Grocery Stores hardest**, with pass rates 
  falling to ~25% by 2019, while **Schools held up comparatively better**, never 
  falling below ~48%
- **Bakeries suffered the steepest individual decline**, hitting the dataset low of 
  ~18.75% in 2019
- **Post-2019 recovery shows a clear divergence**: childcare facilities (Children's 
  Services, Daycare Above and Under 2 Years, Daycare 2–6 Years) recovered faster and 
  cluster at higher pass rates (65–77% by 2025), while Restaurants, Grocery Stores, 
  and Bakeries recovered more slowly (60–65% by 2025)
- A **secondary, milder dip is visible in 2020** across all facility types, 
  attributable to reduced inspection activity during the COVID-19 pandemic

---

## Contextual Dataset

**Dataset Name:** Chicago Business Licenses  
**Source:** City of Chicago Open Data Portal  
**URL:** https://data.cityofchicago.org/Community-Economic-Development/Business-Licenses/r5kz-chrr  

This dataset contains records of every business license issued by the City of Chicago, 
including all food-related establishments such as restaurants, grocery stores, and 
bakeries, spanning over 1.19 million records across 37 columns. It is a valuable 
contextual dataset because it allows us to compare the **total number of licensed food 
establishments** against the **number actually being inspected** each year — revealing 
whether CDPH inspection coverage has kept pace with the growth of food businesses in 
Chicago. If the number of licensed restaurants is growing while inspection counts remain 
flat or decline, that would suggest a widening inspection coverage gap — a compelling 
public health story for Part 3. Additionally, license start and expiration dates in this 
dataset could help explain why some establishments appear as "Out of Business" in the 
inspection records. For Part 3, rather than loading this full dataset at runtime, we 
will pre-aggregate it down to annual counts of newly issued food-related licenses, 
producing a small CSV well within GitHub's file size limits.

---

## Plot Summary

The Chicago Food Inspections dataset contains **309,328 records** of food establishment 
inspections conducted by the Chicago Department of Public Health (CDPH) between January 
2010 and April 2026. Each row represents a single inspection event. For this dashboard, 
the data was filtered to include only the three meaningful inspection outcomes (Pass, 
Fail, Pass w/ Conditions), valid risk levels (Risk 1–3), and complete years (2010–2025), 
resulting in **246,990 records** across the 8 most frequently inspected facility types.

**Key columns used in this dashboard:**

- `Facility Type` (categorical): Type of establishment — e.g. Restaurant, Grocery 
  Store, School, Bakery, Daycare, Long Term Care
- `Results` (categorical): Inspection outcome — *Pass*, *Fail*, or *Pass w/ Conditions*
- `Risk` (categorical): Risk level assigned to the facility — Risk 1 (High), 
  Risk 2 (Medium), Risk 3 (Low)
- `Inspection Date` (datetime): Date the inspection was conducted, from which `Year` 
  is derived (2010–2025)
- `Violations` (text): Free-text description of any violations found (28% of rows 
  have no violations recorded, representing clean passes)

**Key facts about the data:**

- Restaurants account for the overwhelming majority of inspections (approx. 175,000 out of 
  246,990 filtered records) — nearly 6× more than the next largest group, Grocery 
  Stores (roughly 31,000)
- In aggregate across all facility types, roughly half of all inspections result in a 
  Pass, with Fail and Pass w/ Conditions making up the remaining share — visible in 
  the default state of the donut chart
- A sharp, universal drop in pass rates occurred in **2018–2019** across ALL facility 
  types due to CDPH redefining its violation categories in July 2018 — the most 
  dramatic structural shift visible in the line chart
- The 2018 crash hit Restaurants and Grocery Stores the hardest, with pass rates 
  falling to ~25% by 2019, while Schools held up comparatively better, never falling 
  below ~48%
- Bakeries suffered the steepest individual decline, hitting the dataset low of 
  **~18.75%** in 2019
- Post-2019 recovery shows a clear divergence: childcare facilities (Children's 
  Services, Daycare Above and Under 2 Years, Daycare 2–6 Years) recovered faster and 
  cluster at higher pass rates (65–77% by 2025), while Restaurants, Grocery Stores, 
  and Bakeries recovered more slowly and sit lower (60–65% by 2025)
- A secondary, milder dip is visible across all facility types in **2020**, 
  attributable to the COVID-19 pandemic reducing inspection activity.